# Thêm Thư Viện

In [3]:
import pyodbc
import pandas as pd

# Tạo kết nối

In [4]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)
conn_dwh_lib = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=dwh_lib;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2024;'
)


# ETL bảng Dim_Date

### Xóa dữ liệu bảng cũ

In [10]:
# xóa dữ liệu trước khi load vào bảng 
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Date"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

### Đọc dữ liệu

In [11]:
Dim_Date_data = pd.read_csv("ETL_Dim_Date.csv")
print(Dim_Date_data)

       Date_key   Full_date                   Date_text   Day_name  Week  \
0      19700101    1/1/1970     Thursday-January 1-1970   Thursday     1   
1      19700102    1/2/1970       Friday-January 2-1970     Friday     1   
2      19700103    1/3/1970     Saturday-January 3-1970   Saturday     1   
3      19700104    1/4/1970       Sunday-January 4-1970     Sunday     1   
4      19700105    1/5/1970       Monday-January 5-1970     Monday     2   
...         ...         ...                         ...        ...   ...   
29215  20491227  12/27/2049     Monday-December 27-2049     Monday    52   
29216  20491228  12/28/2049    Tuesday-December 28-2049    Tuesday    52   
29217  20491229  12/29/2049  Wednesday-December 29-2049  Wednesday    52   
29218  20491230  12/30/2049   Thursday-December 30-2049   Thursday    52   
29219  20491231  12/31/2049     Friday-December 31-2049     Friday    52   

       Day_of_week  Month  Quarter  Year  Day  
0                4      1        1  197

In [12]:

for index, row in Dim_Date_data.iterrows():
    cursor = conn_dwh_lib.cursor()
    insert_query = """
    INSERT INTO DIM_Date (Date_key, Full_date, Date_text, Day, Week_of_quarter, Month, Quarter, Year, Day_of_week, Day_name)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    cursor.execute(insert_query, row['Date_key'], row['Full_date'], row['Date_text'], row['Day'], 
                   row['Week'], row['Month'], row['Quarter'], row['Year'], row['Day_of_week'], row['Day_name'])
    conn_dwh_lib.commit()
    cursor.close()

# ETL bảng Dim_Dan_Toc

In [5]:
# xóa dữ liệu trước khi load vào bảng 
cursor = conn_dwh_lib.cursor()
truncate_query = "DELETE FROM DIM_Dan_toc"
cursor.execute(truncate_query)
conn_dwh_lib.commit()
cursor.close()

In [6]:
# Câu truy vấn để lấy dữ liệu từ bảng Dan_toc
query_dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "

# Đọc dữ liệu từ bảng Dan_toc vào DataFrame
df = pd.read_sql(query_dan_toc, conn_libol)

# Hiển thị DataFrame
print(df)

    Id  Dan_toc
0    1     Kinh
1    2    Mường
2    3      Tày
3    4     Thái
4    5      Hoa
..  ..      ...
68  71     Ê Đê
69  72      Thổ
70  73    Kờ Ho
71  74     Jrai
72  75  Châu mạ

[73 rows x 2 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_21168\4132415106.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_dan_toc, conn_libol)


In [7]:
for index, row in df.iterrows():
    cursor_dwh = conn_dwh_lib.cursor()
    insert_query = """
    INSERT INTO DIM_Dan_toc (ID_dan_toc, Dan_toc) VALUES (?, ?)
    """
    cursor_dwh.execute(insert_query, row['Id'], row['Dan_toc'])
    conn_dwh_lib.commit()
    cursor_dwh.close()

In [13]:

cursor_dan_toc = conn.cursor() # Tạo đối tượng cursor để thực hiện truy vấn

query = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
cursor_dan_toc.execute(query)

data = cursor_dan_toc.fetchall()

columns = [column[0] for column in cursor_dan_toc.description]

data_frame = pd.DataFrame.from_records(data, columns=columns)


In [14]:
# Hiển thị dữ liệu
print(data_frame.head())


   Id Dan_toc
0   1    Kinh
1   2   Mường
2   3     Tày
3   4    Thái
4   5     Hoa
